In [4]:
# Minimal setup imports to restore kernel state after restart
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
print('Setup imports loaded')


Setup imports loaded


In [5]:
!pip install pandas matplotlib seaborn



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
print("Current directory:", os.getcwd())
print("Files here:", os.listdir())


Current directory: c:\Users\camil\Desktop\Python project Egypt\code
Files here: ['01_data_cleaning.ipynb', '02_data_exploration.ipynb', '03_data_analysis.ipynb', 'peaceful_vs_violent_by_dataset.html', 'peaceful_vs_violent_protests.html']


First, I want to see how many elements my first dataset has (ACLED_egypt2011), and check that the timeframe is correct.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

# Works whether the notebook runs from project root or the code/ folder.
data_path = Path("..") / "raw_data" / "ACLED_egypt2011.csv"
if not data_path.exists():
    data_path = Path("raw_data") / "ACLED_egypt2011.csv"

df_egypt2011 = pd.read_csv(data_path)
print(f"Loaded {len(df_egypt2011):,} rows and {df_egypt2011.shape[1]} columns from {data_path}")
df_egypt2011.head()

Loaded 2,607 rows and 32 columns from ..\raw_data\ACLED_egypt2011.csv


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,population_best
0,EGY2776,2013-07-02,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Muslim Brotherhood,Protesters,...,31.1143,30.9401,1,Egypt Independent,National,"In Kafr al-Sheikh, Nile Delta governorate, hun...",0,NaN,1618528752,NaN
1,EGY2748,2013-07-01,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,29.3099,30.8418,1,Egypt Independent,National,"In al-Massala Sqaure, dozens of opponents prot...",0,NaN,1618528749,NaN
2,EGY2749,2013-07-01,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Muslim Brotherhood,Protesters,...,30.0310,31.1111,1,Egypt Independent,National,Dozens of members of the Muslim Brotherhood be...,0,NaN,1618528749,NaN
3,EGY2665,2013-06-19,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,30.5526,31.0090,1,Egypt Independent,National,Protesters continued their sit-in for a third ...,0,NaN,1618528740,NaN
4,EGY2659,2013-06-18,2013,1,Demonstrations,Riots,Violent demonstration,Rioters (Egypt),NaN,Rioters,...,30.9761,31.1669,1,Egypt Independent,National,"In a separate incident, Kafr al-Sheikh Governo...",0,NaN,1618528739,NaN


Now, since I want to focus on domestic protests against the government, I decided to filter out protests events related to the Israeli-Palestinian conflict. Therefore, I will exclude from the dataset the protests occurred that include the words 'israel', 'israeli', 'palestine, 'palestinian'.

In [8]:
import re

keywords = ["israel", "israeli", "palestine", "palestinian"]
document_text = " ".join(df_egypt2011.fillna("").astype(str).to_numpy().ravel()).lower()

whole_word_counts = {
    kw: len(re.findall(rf"\b{re.escape(kw)}\b", document_text))
    for kw in keywords
}

print("Whole-word occurrences:")
for kw, count in whole_word_counts.items():
    print(f"{kw}: {count}")

print(f"Total whole-word matches: {sum(whole_word_counts.values())}")

Whole-word occurrences:
israel: 8
israeli: 51
palestine: 7
palestinian: 10
Total whole-word matches: 76


In [9]:
import re

exclude_keywords = ["israel", "israeli", "palestine", "palestinian"]
exclude_pattern = r"\b(?:" + "|".join(map(re.escape, exclude_keywords)) + r")\b"

# Combine each row into one string, then mark rows that contain any exclusion keyword.
row_text = df_egypt2011.fillna("").astype(str).agg(" ".join, axis=1)
mask_excluded = row_text.str.contains(exclude_pattern, case=False, na=False, regex=True)

df_egypt2011_clean = df_egypt2011.loc[~mask_excluded].copy()

print(f"Original events: {len(df_egypt2011):,}")
print(f"Removed events: {mask_excluded.sum():,}")
print(f"Remaining events: {len(df_egypt2011_clean):,}")

df_egypt2011_clean.head()

Original events: 2,607
Removed events: 43
Remaining events: 2,564


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,population_best
0,EGY2776,2013-07-02,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Muslim Brotherhood,Protesters,...,31.1143,30.9401,1,Egypt Independent,National,"In Kafr al-Sheikh, Nile Delta governorate, hun...",0,NaN,1618528752,NaN
1,EGY2748,2013-07-01,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,29.3099,30.8418,1,Egypt Independent,National,"In al-Massala Sqaure, dozens of opponents prot...",0,NaN,1618528749,NaN
2,EGY2749,2013-07-01,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Muslim Brotherhood,Protesters,...,30.0310,31.1111,1,Egypt Independent,National,Dozens of members of the Muslim Brotherhood be...,0,NaN,1618528749,NaN
3,EGY2665,2013-06-19,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,30.5526,31.0090,1,Egypt Independent,National,Protesters continued their sit-in for a third ...,0,NaN,1618528740,NaN
4,EGY2659,2013-06-18,2013,1,Demonstrations,Riots,Violent demonstration,Rioters (Egypt),NaN,Rioters,...,30.9761,31.1669,1,Egypt Independent,National,"In a separate incident, Kafr al-Sheikh Governo...",0,NaN,1618528739,NaN


After this, I want to check again the number of elements of the dataset.

In [10]:
# Summary of the cleaned dataset
rows, cols = df_egypt2011_clean.shape
print(f"Rows: {rows:,}")
print(f"Columns: {cols}")

print("\nColumn titles:")
print(df_egypt2011_clean.columns.tolist())

print("\nRow titles (index) preview:")
print(df_egypt2011_clean.index.tolist()[:20])

Rows: 2,564
Columns: 32

Column titles:
['event_id_cnty', 'event_date', 'year', 'time_precision', 'disorder_type', 'event_type', 'sub_event_type', 'actor1', 'assoc_actor_1', 'inter1', 'actor2', 'assoc_actor_2', 'inter2', 'interaction', 'civilian_targeting', 'iso', 'region', 'country', 'admin1', 'admin2', 'admin3', 'location', 'latitude', 'longitude', 'geo_precision', 'source', 'source_scale', 'notes', 'fatalities', 'tags', 'timestamp', 'population_best']

Row titles (index) preview:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


I want to check that all the event protests have latitude and longitude values, before conducting my analysis. In case there are missing values, I will exclude those rows.

In [11]:
# Check whether protest events have city/location and geolocation
protests = df_egypt2011_clean[df_egypt2011_clean["event_type"].astype(str).str.lower() == "protests"].copy()

total_protests = len(protests)
missing_location = protests["location"].isna() | (protests["location"].astype(str).str.strip() == "")
missing_lat = protests["latitude"].isna()
missing_lon = protests["longitude"].isna()
missing_geo = missing_lat | missing_lon

print(f"Total protest events: {total_protests:,}")
print(f"Missing city/location: {missing_location.sum():,}")
print(f"Missing latitude: {missing_lat.sum():,}")
print(f"Missing longitude: {missing_lon.sum():,}")
print(f"Missing geolocation (lat or lon): {missing_geo.sum():,}")

if missing_location.sum() == 0 and missing_geo.sum() == 0:
    print("All protest events have both location and geolocation.")
else:
    print("Not all protest events have complete location/geolocation data.")

Total protest events: 1,780
Missing city/location: 0
Missing latitude: 0
Missing longitude: 0
Missing geolocation (lat or lon): 0
All protest events have both location and geolocation.


All events protests have latitude and longitude values, therefore I will not have to exclude any rows.

Now I want to do the same data cleaning, but for my second dataset "ACLED_alsisi", which takes into account Egyptian protests from 2019 to 2020.

First, I want to see how many elements my second dataset has.

In [12]:
import re
from pathlib import Path

# Step 1: Load ACLED_alsisi dataset
alsisi_path = Path("..") / "raw_data" / "ACLED_alsisi.csv"
if not alsisi_path.exists():
    alsisi_path = Path("raw_data") / "ACLED_alsisi.csv"

df_alsisi = pd.read_csv(alsisi_path)
print(f"Loaded ACLED_alsisi: {len(df_alsisi):,} rows, {df_alsisi.shape[1]} columns")
df_alsisi.head()

Loaded ACLED_alsisi: 153 rows, 32 columns


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,population_best
0,EGY10042,2019-09-22,2019,1,Political violence; Demonstrations,Protests,Excessive force against protesters,Protesters (Egypt),NaN,Protesters,...,29.9737,32.5263,1,AFP; EuroNews; Facebook,New media-International,"During the night of 22 Sep. 2019, security for...",0,crowd size=200,1618528470,84556
1,EGY10634,2019-12-20,2019,2,Demonstrations,Protests,Protest with intervention,Protesters (Egypt),Labor Group (Egypt),Protesters,...,30.2964,31.7463,1,FJ Portal,National,"On 20 December 2019, police intervened in a pr...",0,crowd size=no report,1618528543,16673
2,EGY10657,2019-12-15,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,31.0392,30.4691,1,FJ Portal,National,"On 15 December 2019, protesters organizing und...",0,crowd size=no report,1618528545,122636
3,EGY10677,2020-01-21,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Health Workers (Egypt),Protesters,...,28.1099,30.7503,1,Madamasr; El Fagr News,National,"On 21 January 2020, members of the General Ass...",0,crowd size=no report,1618528547,24137
4,EGY10678,2020-01-23,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Journalists (Egypt),Protesters,...,30.0351,31.2140,1,El Fagr News; Masr Alarabia,National,"On 23 January 2020, journalists at Saba7 Newsp...",0,crowd size=no report,1764045636,25670


Once again, just like I did for my first dataset, I want to exclude the words 'israel', 'palestine' , 'palestinian', 'israeli'. I want to make sure that the protests in the dataset are domestic and against the government, rather than related to the bigger regional dynamics.  

In [13]:
# Step 2: Count whole-word mentions of exclusion keywords in full text
keywords_alsisi = ["israel", "israeli", "palestine", "palestinian"]
document_text_alsisi = " ".join(df_alsisi.fillna("").astype(str).to_numpy().ravel()).lower()
whole_word_counts_alsisi = {
    kw: len(re.findall(rf"\b{re.escape(kw)}\b", document_text_alsisi))
    for kw in keywords_alsisi
}

print("Whole-word occurrences in ACLED_alsisi:")
for kw, count in whole_word_counts_alsisi.items():
    print(f"{kw}: {count}")
print(f"Total whole-word matches: {sum(whole_word_counts_alsisi.values())}")

Whole-word occurrences in ACLED_alsisi:
israel: 2
israeli: 0
palestine: 2
palestinian: 1
Total whole-word matches: 5


In [14]:
# Step 3: Remove rows containing exclusion keywords
alsisi_exclude_pattern = r"\b(?:" + "|".join(map(re.escape, keywords_alsisi)) + r")\b"
row_text_alsisi = df_alsisi.fillna("").astype(str).agg(" ".join, axis=1)
mask_excluded_alsisi = row_text_alsisi.str.contains(alsisi_exclude_pattern, case=False, na=False, regex=True)
df_alsisi_clean = df_alsisi.loc[~mask_excluded_alsisi].copy()

print("Filtering summary:")
print(f"Original events: {len(df_alsisi):,}")
print(f"Removed events: {mask_excluded_alsisi.sum():,}")
print(f"Remaining events: {len(df_alsisi_clean):,}")
df_alsisi_clean.head()

Filtering summary:
Original events: 153
Removed events: 2
Remaining events: 151


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,population_best
0,EGY10042,2019-09-22,2019,1,Political violence; Demonstrations,Protests,Excessive force against protesters,Protesters (Egypt),NaN,Protesters,...,29.9737,32.5263,1,AFP; EuroNews; Facebook,New media-International,"During the night of 22 Sep. 2019, security for...",0,crowd size=200,1618528470,84556
1,EGY10634,2019-12-20,2019,2,Demonstrations,Protests,Protest with intervention,Protesters (Egypt),Labor Group (Egypt),Protesters,...,30.2964,31.7463,1,FJ Portal,National,"On 20 December 2019, police intervened in a pr...",0,crowd size=no report,1618528543,16673
2,EGY10657,2019-12-15,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,31.0392,30.4691,1,FJ Portal,National,"On 15 December 2019, protesters organizing und...",0,crowd size=no report,1618528545,122636
3,EGY10677,2020-01-21,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Health Workers (Egypt),Protesters,...,28.1099,30.7503,1,Madamasr; El Fagr News,National,"On 21 January 2020, members of the General Ass...",0,crowd size=no report,1618528547,24137
4,EGY10678,2020-01-23,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Journalists (Egypt),Protesters,...,30.0351,31.2140,1,El Fagr News; Masr Alarabia,National,"On 23 January 2020, journalists at Saba7 Newsp...",0,crowd size=no report,1764045636,25670


Now that I have deleted these events, I want to check again the elements in the dataset as well as ensure that the timeframe is correct.

In [15]:
# Step 4: Cleaned dataset shape
rows_alsisi, cols_alsisi = df_alsisi_clean.shape
print(f"Rows: {rows_alsisi:,}")
print(f"Columns: {cols_alsisi}")

print("\nColumn titles:")
print(df_alsisi_clean.columns.tolist())

Rows: 151
Columns: 32

Column titles:
['event_id_cnty', 'event_date', 'year', 'time_precision', 'disorder_type', 'event_type', 'sub_event_type', 'actor1', 'assoc_actor_1', 'inter1', 'actor2', 'assoc_actor_2', 'inter2', 'interaction', 'civilian_targeting', 'iso', 'region', 'country', 'admin1', 'admin2', 'admin3', 'location', 'latitude', 'longitude', 'geo_precision', 'source', 'source_scale', 'notes', 'fatalities', 'tags', 'timestamp', 'population_best']


I want to check that all protests have latitude and longitude values, so there is nothing missing when I will start doing my analysis.

In [16]:
# Step 5: Geolocation completeness for protest events
protests_alsisi = df_alsisi_clean[df_alsisi_clean["event_type"].astype(str).str.lower() == "protests"].copy()
missing_location_alsisi = protests_alsisi["location"].isna() | (protests_alsisi["location"].astype(str).str.strip() == "")
missing_lat_alsisi = protests_alsisi["latitude"].isna()
missing_lon_alsisi = protests_alsisi["longitude"].isna()
missing_geo_alsisi = missing_lat_alsisi | missing_lon_alsisi

print(f"Total protest events: {len(protests_alsisi):,}")
print(f"Missing city/location: {missing_location_alsisi.sum():,}")
print(f"Missing latitude: {missing_lat_alsisi.sum():,}")
print(f"Missing longitude: {missing_lon_alsisi.sum():,}")
print(f"Missing geolocation (lat or lon): {missing_geo_alsisi.sum():,}")

Total protest events: 137
Missing city/location: 0
Missing latitude: 0
Missing longitude: 0
Missing geolocation (lat or lon): 0


I will start my exploration by plotting a chart where I can see the monthly quantity of the 4 different protest subtypes (peaceful protest, protest with intervention, violent demonstration, excessive use against protesters), for both the two datasets. I will make the graphs interactive.

In [17]:
from pathlib import Path
import re
import pandas as pd
import plotly.express as px


def load_clean_acled(csv_name):
    path = Path("..") / "raw_data" / csv_name
    if not path.exists():
        path = Path("raw_data") / csv_name

    df = pd.read_csv(path)

    # Keep domestic-focus cleaning used in your notebook.
    pattern = r"\b(?:israel|israeli|palestine|palestinian)\b"
    row_text = df.fillna("").astype(str).agg(" ".join, axis=1)
    df = df.loc[~row_text.str.contains(pattern, case=False, regex=True)].copy()
    df = df.loc[~df["sub_event_type"].astype(str).str.strip().str.lower().eq("mob violence")].copy()

    df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
    return df.dropna(subset=["event_date", "sub_event_type"])


def plot_top4_subtypes_by_month(df, title):
    top4 = df["sub_event_type"].value_counts().head(4).index.tolist()

    monthly = (
        df[df["sub_event_type"].isin(top4)]
        .assign(month=df["event_date"].dt.to_period("M").astype(str))
        .groupby(["month", "sub_event_type"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )

    # Custom colors requested by user.
    color_map = {
        "Protest with intervention": "#000000",  # black
        "Excessive force against protesters": "#808080",  # grey
        "Peaceful protest": "#ff7f0e",  # orange
        "Violent demonstration": "#d62728",  # red
    }

    fig = px.bar(
        monthly,
        x="month",
        y="count",
        color="sub_event_type",
        color_discrete_map=color_map,
        barmode="stack",
        title=title,
    )
    fig.update_layout(
        xaxis_title="Month",
        yaxis_title="Number of events",
        legend_title="Sub-event type",
        bargap=0,
        bargroupgap=0,
    )
    fig.update_traces(
        hovertemplate="Month: %{x}<br>Subtype: %{fullData.name}<br>Count: %{y}<extra></extra>",
        marker_line_width=0,
    )
    fig.show()


# Cell 26: ACLED_egypt2011 interactive monthly chart
df_2011_clean = load_clean_acled("ACLED_egypt2011.csv")
plot_top4_subtypes_by_month(df_2011_clean, "Demonstration Events in Egypt by month (Arab Spring)")

In [18]:
# Cell 26: ACLED_alsisi chart (reuses helper functions from Cell 25)
df_alsisi_clean = load_clean_acled("ACLED_alsisi.csv")
plot_top4_subtypes_by_month(df_alsisi_clean, "Demonstration Events in Egypt by month (2019-2020)")

I will now do the same, but per year.

In [19]:
# Yearly interactive stacked bars for top 4 sub-event types (ACLED_egypt2011)

def plot_top4_subtypes_by_year(df, title):
    top4 = df["sub_event_type"].value_counts().head(4).index.tolist()

    yearly = (
        df[df["sub_event_type"].isin(top4)]
        .assign(year=df["event_date"].dt.year.astype(str))
        .groupby(["year", "sub_event_type"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )

    color_map = {
        "Protest with intervention": "#000000",  # black
        "Excessive force against protesters": "#808080",  # grey
        "Peaceful protest": "#ff7f0e",  # orange
        "Violent demonstration": "#d62728",  # red
    }

    fig = px.bar(
        yearly,
        x="year",
        y="count",
        color="sub_event_type",
        color_discrete_map=color_map,
        barmode="stack",
        title=title,
    )
    fig.update_layout(
        xaxis_title="Year",
        yaxis_title="Number of events",
        legend_title="Sub-event type",
        bargap=0.25,
        bargroupgap=0,
    )
    fig.update_traces(
        hovertemplate="Year: %{x}<br>Subtype: %{fullData.name}<br>Count: %{y}<extra></extra>",
        marker_line_width=0,
    )
    fig.show()


df_2011_clean = load_clean_acled("ACLED_egypt2011.csv")
plot_top4_subtypes_by_year(df_2011_clean, "Demonstration Events in Egypt by year (Arab Spring)")

In [20]:
# Yearly histogram-style bars for top 4 sub-event types (ACLED_alsisi)
df_alsisi_clean = load_clean_acled("ACLED_alsisi.csv")
plot_top4_subtypes_by_year(df_alsisi_clean, "Demonstration Events in Egypt by year (2019-2020)")

In [44]:
# Cleaner yearly percentage chart with custom panel titles
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

subtypes = [
    'Peaceful protest',
    'Protest with intervention',
    'Excessive force against protesters',
    'Violent demonstration'
]

subtype_colors = {
    'Peaceful protest': '#ff7f0e',
    'Protest with intervention': '#000000',
    'Excessive force against protesters': '#808080',
    'Violent demonstration': '#d62728'
}

def build_yearly_share(df):
    sub_col = 'sub_event_type' if 'sub_event_type' in df.columns else 'SUB_EVENT_TYPE'
    date_col = 'event_date' if 'event_date' in df.columns else 'EVENT_DATE'

    work = df[[sub_col, date_col]].copy()
    work[date_col] = pd.to_datetime(work[date_col], errors='coerce')
    work = work.dropna(subset=[date_col])
    work = work[work[sub_col].isin(subtypes)].copy()
    work['year'] = work[date_col].dt.year.astype(str)

    counts = (
        work.groupby(['year', sub_col])
        .size()
        .rename('count')
        .reset_index()
        .rename(columns={sub_col: 'sub_event_type'})
    )

    if counts.empty:
        return counts

    totals = counts.groupby('year', as_index=False)['count'].sum().rename(columns={'count': 'year_total'})
    counts = counts.merge(totals, on='year', how='left')
    counts['percent'] = counts['count'] / counts['year_total'] * 100
    return counts

arab_spring = build_yearly_share(df_egypt2011_clean)
egypt_2019_2020 = build_yearly_share(df_alsisi_clean)

fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    subplot_titles=('Arab Spring', '2019-2020 protests'),
)

for col, data in [(1, arab_spring), (2, egypt_2019_2020)]:
    if data.empty:
        continue

    years = sorted(data['year'].unique(), key=lambda value: int(value))
    for subtype in subtypes:
        subset = data[data['sub_event_type'] == subtype].set_index('year').reindex(years).fillna(0).reset_index()
        fig.add_trace(
            go.Bar(
                x=subset['year'],
                y=subset['percent'],
                name=subtype,
                marker_color=subtype_colors[subtype],
                legendgroup=subtype,
                showlegend=(col == 1),
                hovertemplate='Year: %{x}<br>Subtype: %{fullData.name}<br>Share: %{y:.1f}%<extra></extra>',
            ),
            row=1,
            col=col,
        )

fig.update_layout(
    barmode='stack',
    template='simple_white',
    title='Demonstration events in Egypt by year (%)',
    height=520,
    margin=dict(l=40, r=20, t=80, b=40),
    legend_title_text='Sub-event type',
)
fig.update_yaxes(range=[0, 100], title_text='Percentage of events')
fig.update_xaxes(title_text='Year', tickangle=0)
fig.show()

In [ ]:
# Demonstration events in Egypt by year (percentage), split by dataset
import pandas as pd
import plotly.express as px

subtypes = [
    'Peaceful protest',
    'Protest with intervention',
    'Excessive force against protesters',
    'Violent demonstration'
]

subtype_colors = {
    'Peaceful protest': '#ff7f0e',
    'Protest with intervention': '#000000',
    'Excessive force against protesters': '#808080',
    'Violent demonstration': '#d62728'
}

records = []
for dataset_name, df in [
    ('Arab Spring (2011-2013)', df_egypt2011_clean),
    ('2019-2020', df_alsisi_clean)
]:
    sub_col = 'sub_event_type' if 'sub_event_type' in df.columns else 'SUB_EVENT_TYPE'
    date_col = 'event_date' if 'event_date' in df.columns else 'EVENT_DATE'

    work = df[[sub_col, date_col]].copy()
    work[date_col] = pd.to_datetime(work[date_col], errors='coerce')
    work = work.dropna(subset=[date_col])
    work = work[work[sub_col].isin(subtypes)].copy()
    work['year'] = work[date_col].dt.year.astype(str)

    yearly_counts = (
        work.groupby(['year', sub_col])
        .size()
        .rename('count')
        .reset_index()
        .rename(columns={sub_col: 'sub_event_type'})
    )

    if yearly_counts.empty:
        continue

    yearly_totals = yearly_counts.groupby('year')['count'].sum().rename('year_total').reset_index()
    yearly_counts = yearly_counts.merge(yearly_totals, on='year', how='left')
    yearly_counts['percent'] = (yearly_counts['count'] / yearly_counts['year_total']) * 100
    yearly_counts['dataset'] = dataset_name

    records.append(yearly_counts[['dataset', 'year', 'sub_event_type', 'percent']])

if not records:
    print('No valid yearly subtype data found for plotting.')
else:
    plot_df_year_pct = pd.concat(records, ignore_index=True)

    fig = px.bar(
        plot_df_year_pct,
        x='year',
        y='percent',
        color='sub_event_type',
        barmode='stack',
        facet_col='dataset',
        category_orders={
            'sub_event_type': subtypes,
            'dataset': ['Arab Spring (2011-2013)', '2019-2020']
        },
        color_discrete_map=subtype_colors,
        title='Demonstration Events in Egypt by Year (%)',
        labels={
            'year': 'Year',
            'percent': 'Percentage of events',
            'sub_event_type': 'Sub-event type',
            'dataset': 'Dataset'
        }
    )
    fig.update_yaxes(range=[0, 100])
    fig.update_layout(legend_title_text='Sub-event type')
    fig.show()

In [22]:
# Demonstration events in Egypt by dataset across the 4 key sub-event types
# One stacked bar per dataset, using the notebook's established subtype colors.
import pandas as pd
import plotly.express as px

subtypes = [
    'Peaceful protest',
    'Protest with intervention',
    'Excessive force against protesters',
    'Violent demonstration'
]

subtype_colors = {
    'Peaceful protest': '#ff7f0e',
    'Protest with intervention': '#000000',
    'Excessive force against protesters': '#808080',
    'Violent demonstration': '#d62728'
}

records = []
for dataset_name, df in [
    ('Arab Spring (2011-2013)', df_egypt2011_clean),
    ('2019-2020', df_alsisi_clean)
]:
    sub_col = 'sub_event_type' if 'sub_event_type' in df.columns else 'SUB_EVENT_TYPE'
    counts = (
        df[df[sub_col].isin(subtypes)][sub_col]
        .value_counts()
        .reindex(subtypes, fill_value=0)
    )
    for subtype, count in counts.items():
        records.append({
            'dataset': dataset_name,
            'sub_event_type': subtype,
            'count': int(count)
        })

plot_df = pd.DataFrame(records)

fig = px.bar(
    plot_df,
    x='dataset',
    y='count',
    color='sub_event_type',
    barmode='stack',
    text='count',
    category_orders={'sub_event_type': subtypes},
    color_discrete_map=subtype_colors,
    title='Demonstration Events in Egypt by Dataset and Sub-Event Type',
    labels={
        'dataset': 'Dataset',
        'count': 'Number of events',
        'sub_event_type': 'Sub-event type'
    }
)
fig.update_traces(textposition='inside')
fig.update_layout(legend_title_text='Sub-event type')
fig.show()


In [23]:
# Demonstration events in Egypt by dataset across the 4 key sub-event types (percentage)
# One stacked bar per dataset in %, using the same subtype colors.
import pandas as pd
import plotly.express as px

subtypes = [
    'Peaceful protest',
    'Protest with intervention',
    'Excessive force against protesters',
    'Violent demonstration'
]

subtype_colors = {
    'Peaceful protest': '#ff7f0e',
    'Protest with intervention': '#000000',
    'Excessive force against protesters': '#808080',
    'Violent demonstration': '#d62728'
}

records = []
for dataset_name, df in [
    ('Arab Spring (2011-2013)', df_egypt2011_clean),
    ('2019-2020', df_alsisi_clean)
]:
    sub_col = 'sub_event_type' if 'sub_event_type' in df.columns else 'SUB_EVENT_TYPE'
    counts = (
        df[df[sub_col].isin(subtypes)][sub_col]
        .value_counts()
        .reindex(subtypes, fill_value=0)
    )

    total = int(counts.sum())
    for subtype, count in counts.items():
        pct = (count / total * 100) if total > 0 else 0
        records.append({
            'dataset': dataset_name,
            'sub_event_type': subtype,
            'percent': pct
        })

plot_df_pct = pd.DataFrame(records)

fig = px.bar(
    plot_df_pct,
    x='dataset',
    y='percent',
    color='sub_event_type',
    barmode='stack',
    text='percent',
    category_orders={'sub_event_type': subtypes},
    color_discrete_map=subtype_colors,
    title='Demonstration Events in Egypt by Dataset and Sub-Event Type (%)',
    labels={
        'dataset': 'Dataset',
        'percent': 'Percentage of events',
        'sub_event_type': 'Sub-event type'
    }
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='inside')
fig.update_layout(
    yaxis=dict(range=[0, 100]),
    legend_title_text='Sub-event type'
)
fig.show()


In [24]:
# Interactive chart: month-by-month protest counts (ACLED_egypt2011)

def plot_protest_counts_by_month(df, title):
    protests_df = df[df["event_type"].astype(str).str.lower() == "protests"].copy()

    monthly_counts = (
        protests_df
        .assign(month=protests_df["event_date"].dt.to_period("M").astype(str))
        .groupby("month", as_index=False)
        .size()
        .rename(columns={"size": "protest_count"})
        .sort_values("month")
    )

    fig = px.line(
        monthly_counts,
        x="month",
        y="protest_count",
        title=title,
        markers=True,
    )
    fig.update_traces(
        line_color="#1f77b4",
        hovertemplate="Month: %{x}<br>Protests: %{y}<extra></extra>",
    )
    fig.update_layout(
        xaxis_title="Month",
        yaxis_title="Number of events",
    )
    fig.show()


plot_protest_counts_by_month(
    df_2011_clean,
    "Demonstration Events in Egypt by month (Arab Spring)",
)

In [25]:
# Interactive chart: month-by-month protest counts (ACLED_alsisi)
plot_protest_counts_by_month(
    df_alsisi_clean,
    "Demonstration Events in Egypt by month (2019-2020)",
)

In [26]:
from pathlib import Path
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def load_clean_acled(csv_name):
    path = Path("..") / "raw_data" / csv_name
    if not path.exists():
        path = Path("raw_data") / csv_name

    df = pd.read_csv(path)

    # Keep domestic-focus cleaning used in your notebook.
    pattern = r"\b(?:israel|israeli|palestine|palestinian)\b"
    row_text = df.fillna("").astype(str).agg(" ".join, axis=1)
    df = df.loc[~row_text.str.contains(pattern, case=False, regex=True)].copy()
    df = df.loc[~df["sub_event_type"].astype(str).str.strip().str.lower().eq("mob violence")].copy()

    df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
    return df.dropna(subset=["event_date", "sub_event_type"])


def plot_protest_subtypes_over_time(df, title):
    protests_df = df[df["event_type"].astype(str).str.lower() == "protests"].copy()
    protests_df["month"] = protests_df["event_date"].dt.to_period("M").astype(str)

    monthly = protests_df.groupby(["month", "sub_event_type"]).size().unstack(fill_value=0).sort_index()

    total_counts = monthly.sum(axis=1)
    excessive_force = monthly.get("Excessive force against protesters", pd.Series(0, index=monthly.index))
    intervention = monthly.get("Protest with intervention", pd.Series(0, index=monthly.index))

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=("Total demonstrations", "Excessive force vs. protest with intervention"),
    )

    fig.add_trace(
        go.Scatter(x=total_counts.index, y=total_counts.values, mode="lines+markers", name="Total protests", line=dict(color="#1f77b4")),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=excessive_force.index, y=excessive_force.values, mode="lines+markers", name="Excessive force against protesters", line=dict(color="#808080")),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=intervention.index, y=intervention.values, mode="lines+markers", name="Protest with intervention", line=dict(color="#000000")),
        row=2,
        col=1,
    )

    fig.update_layout(title=title, xaxis2_title="Month", yaxis_title="Event counts", yaxis2_title="Subtype counts", legend_title="Series")
    fig.update_traces(hovertemplate="Month: %{x}<br>Count: %{y}<extra></extra>")
    fig.show()


# Cell 26: ACLED_egypt2011 interactive monthly chart
df_2011_clean = load_clean_acled("ACLED_egypt2011.csv")
plot_protest_subtypes_over_time(
    df_2011_clean,
    "Demonstration Events in Egypt by month (Arab Spring)",
)

In [27]:
# Two-panel monthly protest trends (ACLED_alsisi)
plot_protest_subtypes_over_time(
    df_alsisi_clean,
    "Demonstration Events in Egypt by month (2019-2020)",
)

In [28]:
# Monthly percentage of protests by subtype (ACLED_egypt2011)

def plot_protest_percentages_by_month(df, title):
    protests_df = df[df["event_type"].astype(str).str.lower() == "protests"].copy()
    protests_df["month"] = protests_df["event_date"].dt.to_period("M").astype(str)

    monthly_total = protests_df.groupby("month").size().rename("total protests")
    monthly_excessive = (
        protests_df[protests_df["sub_event_type"].astype(str).str.lower() == "excessive force against protesters"]
        .groupby("month")
        .size()
        .rename("excessive force against protesters")
    )
    monthly_intervention = (
        protests_df[protests_df["sub_event_type"].astype(str).str.lower() == "protest with intervention"]
        .groupby("month")
        .size()
        .rename("protest with intervention")
    )

    chart_df = pd.concat([monthly_total, monthly_excessive, monthly_intervention], axis=1).fillna(0).reset_index()
    chart_df = chart_df.sort_values("month")

    chart_df["excessive state force %"] = (chart_df["excessive force against protesters"] / chart_df["total protests"]) * 100
    chart_df["state intervention %"] = (chart_df["protest with intervention"] / chart_df["total protests"]) * 100
    chart_df["other protest subtypes %"] = 100 - chart_df["excessive state force %"] - chart_df["state intervention %"]

    long_df = chart_df.melt(
        id_vars="month",
        value_vars=["excessive state force %", "state intervention %", "other protest subtypes %"],
        var_name="series",
        value_name="percent",
    )

    fig = px.bar(
        long_df,
        x="month",
        y="percent",
        color="series",
        title=title,
        barmode="stack",
        color_discrete_map={
            "excessive state force %": "#808080",
            "state intervention %": "#000000",
            "other protest subtypes %": "#ff7f0e",
        },
    )
    fig.update_traces(hovertemplate="Month: %{x}<br>Series: %{fullData.name}<br>Percent: %{y:.1f}%<extra></extra>")
    fig.update_layout(
        xaxis_title="Month",
        yaxis_title="Percent of total demonstrations",
        legend_title="Series",
        bargap=0.2,
        bargroupgap=0,
    )
    fig.show()


plot_protest_percentages_by_month(
    df_2011_clean,
    "Proportion of state repression by month in % (Arab Spring)",
)

In [46]:
# Monthly percentage of protests by subtype (ACLED_alsisi)
import pandas as pd
import plotly.express as px


def plot_protest_percentages_by_month(df, title):
    protests_df = df[df["event_type"].astype(str).str.lower() == "protests"].copy()
    protests_df["month"] = protests_df["event_date"].dt.to_period("M").astype(str)

    monthly_total = protests_df.groupby("month").size().rename("total protests")
    monthly_excessive = (
        protests_df[
            protests_df["sub_event_type"].astype(str).str.lower()
            == "excessive force against protesters"
        ]
        .groupby("month")
        .size()
        .rename("excessive force against protesters")
    )
    monthly_intervention = (
        protests_df[
            protests_df["sub_event_type"].astype(str).str.lower()
            == "protest with intervention"
        ]
        .groupby("month")
        .size()
        .rename("protest with intervention")
    )

    chart_df = pd.concat(
        [monthly_total, monthly_excessive, monthly_intervention], axis=1
    ).fillna(0).reset_index()
    chart_df = chart_df.sort_values("month")

    chart_df["excessive state force %"] = (
        chart_df["excessive force against protesters"] / chart_df["total protests"]
    ) * 100
    chart_df["state intervention %"] = (
        chart_df["protest with intervention"] / chart_df["total protests"]
    ) * 100
    chart_df["other protest subtypes %"] = (
        100
        - chart_df["excessive state force %"]
        - chart_df["state intervention %"]
    )

    long_df = chart_df.melt(
        id_vars="month",
        value_vars=[
            "excessive state force %",
            "state intervention %",
            "other protest subtypes %",
        ],
        var_name="series",
        value_name="percent",
    )

    fig = px.bar(
        long_df,
        x="month",
        y="percent",
        color="series",
        title=title,
        barmode="stack",
        color_discrete_map={
            "excessive state force %": "#808080",
            "state intervention %": "#000000",
            "other protest subtypes %": "#ff7f0e",
        },
    )
    fig.update_traces(
        hovertemplate="Month: %{x}<br>Series: %{fullData.name}<br>Percent: %{y:.1f}%<extra></extra>"
    )
    fig.update_layout(
        xaxis_title="Month",
        yaxis_title="Percent of total demonstrations",
        legend_title="Series",
        bargap=0.2,
        bargroupgap=0,
    )
    fig.show()


plot_protest_percentages_by_month(
    df_alsisi_clean,
    "Proportion of state repression by month in % (2019-2020)",
)

In [31]:
# New maps: separate Egypt basemaps for ACLED_egypt2011 and ACLED_alsisi
import pandas as pd
import plotly.express as px

target_subtypes = [
    "Peaceful protest",
    "Protest with intervention",
    "Violent demonstration",
    "Excessive force against protesters",
]

color_map = {
    "Peaceful protest": "#ff7f0e",
    "Protest with intervention": "#000000",
    "Violent demonstration": "#d62728",
    "Excessive force against protesters": "#808080",
}


def prepare_bubbles(df_in, dataset_label):
    if df_in is None or df_in.empty:
        print(f"{dataset_label}: no data available.")
        return None

    df_local = df_in.copy()

    norm_cols = {c.strip().lower(): c for c in df_local.columns}
    pick = lambda *names: next((norm_cols[n] for n in names if n in norm_cols), None)

    event_col = pick("event_type")
    sub_event_col = pick("sub_event_type")
    lat_col = pick("latitude", "lat")
    lon_col = pick("longitude", "lon", "lng")
    loc_col = pick("location", "admin1", "admin2", "admin3")

    if not event_col or not sub_event_col or not lat_col or not lon_col:
        print(f"{dataset_label}: missing required columns.")
        return None

    # Use protests + riots.
    df_local[event_col] = df_local[event_col].astype(str).str.strip().str.lower()
    df_local = df_local[df_local[event_col].isin(["protests", "riots"])].copy()

    df_local[sub_event_col] = df_local[sub_event_col].astype(str).str.strip()
    df_local = df_local[df_local[sub_event_col].isin(target_subtypes)].copy()

    df_local[lat_col] = pd.to_numeric(df_local[lat_col], errors="coerce")
    df_local[lon_col] = pd.to_numeric(df_local[lon_col], errors="coerce")
    df_local = df_local.dropna(subset=[lat_col, lon_col])

    if df_local.empty:
        print(f"{dataset_label}: no protests/riots events for selected subtypes with valid coordinates.")
        return None

    group_cols = [lat_col, lon_col, sub_event_col] + ([loc_col] if loc_col else [])
    bubbles = (
        df_local.groupby(group_cols, dropna=False)
        .size()
        .rename("event_count")
        .reset_index()
    )

    return {
        "label": dataset_label,
        "bubbles": bubbles,
        "lat_col": lat_col,
        "lon_col": lon_col,
        "sub_event_col": sub_event_col,
        "loc_col": loc_col,
        "mapped_events": len(df_local),
    }


def plot_separate_egypt_map(prepped, global_max_count, min_marker_px=4, max_marker_px=42):
    if prepped is None:
        return

    bubbles = prepped["bubbles"].copy()
    label = prepped["label"]
    lat_col = prepped["lat_col"]
    lon_col = prepped["lon_col"]
    sub_event_col = prepped["sub_event_col"]
    loc_col = prepped["loc_col"]

    # Shared monotonic scaling across both maps: larger count -> larger bubble.
    # log1p keeps small/medium count differences visible (e.g., 1 vs 11).
    denom = max(global_max_count, 1)
    bubbles["marker_px"] = min_marker_px + (
        np.log1p(pd.Series(bubbles["event_count"], dtype=float)) / np.log1p(denom)
    ) * (max_marker_px - min_marker_px)

    if label == "ACLED_egypt2011":
        period_label = "Arab Spring"
    elif label == "ACLED_alsisi":
        period_label = "2019-2020"
    else:
        period_label = label

    fig = px.scatter_map(
        bubbles,
        lat=lat_col,
        lon=lon_col,
        size="marker_px",
        color=sub_event_col,
        hover_name=loc_col if loc_col else None,
        hover_data={"event_count": True, "marker_px": False},
        zoom=5.2,
        center={"lat": 26.8, "lon": 30.8},
        height=760,
        title=f"Map of Demonstration Events in Egypt ({period_label})<br><sup>Bubble size proportional to event count, shared scale</sup>",
        color_discrete_map=color_map,
        category_orders={sub_event_col: target_subtypes},
    )

    # Interpret marker_px directly as diameter in px, preventing figure-specific auto-rescaling.
    fig.update_traces(
        marker=dict(
            opacity=0.72,
            sizemode="diameter",
            sizeref=1,
            sizemin=min_marker_px,
        ),
        hovertemplate="Subtype: %{fullData.name}<br>Events: %{customdata[0]}<br>Lat: %{lat:.3f}<br>Lon: %{lon:.3f}<extra></extra>",
    )
    fig.update_layout(
        map_style="carto-positron",
        margin=dict(l=0, r=0, t=70, b=0),
        legend_title_text="Subtype",
    )
    fig.show()

    print(f"{label} mapped events: {prepped['mapped_events']:,}")


prepped_2011 = prepare_bubbles(df_2011_clean, "ACLED_egypt2011")
prepped_alsisi = prepare_bubbles(df_alsisi_clean, "ACLED_alsisi")

valid_prepped = [p for p in [prepped_2011, prepped_alsisi] if p is not None]
if not valid_prepped:
    print("No valid data to map.")
else:
    shared_max_count = max(p["bubbles"]["event_count"].max() for p in valid_prepped)
    print(f"Shared max events per coordinate across both maps: {shared_max_count}")
    plot_separate_egypt_map(prepped_2011, shared_max_count)
    plot_separate_egypt_map(prepped_alsisi, shared_max_count)

Shared max events per coordinate across both maps: 243


ACLED_egypt2011 mapped events: 2,323


ACLED_alsisi mapped events: 147


## Governorate-Level Violence Analysis (Separate by Dataset)
This section groups protests + riots by Egyptian governorate and classifies each governorate into violence levels based on a weighted violence index.

In [33]:
import pandas as pd
import plotly.express as px

TARGET_SUBTYPES = [
    "Peaceful protest",
    "Excessive force against protesters",
    "Violent demonstration",
    "Protest with intervention",
]

SUBTYPE_COLORS = {
    "Peaceful protest": "#ff7f0e",
    "Excessive force against protesters": "#808080",
    "Violent demonstration": "#d62728",
    "Protest with intervention": "#000000",
}


def build_governorate_subtype_percentages(df_in, dataset_label):
    if df_in is None or df_in.empty:
        print(f"{dataset_label}: no data available.")
        return None

    df_local = df_in.copy()
    norm_cols = {c.strip().lower(): c for c in df_local.columns}
    pick = lambda *names: next((norm_cols[n] for n in names if n in norm_cols), None)

    event_col = pick("event_type")
    sub_event_col = pick("sub_event_type")
    gov_col = pick("admin1", "governorate", "admin_1", "location")

    if not event_col or not sub_event_col or not gov_col:
        print(f"{dataset_label}: required columns missing for governorate analysis.")
        return None

    df_local[event_col] = df_local[event_col].astype(str).str.strip().str.lower()
    df_local[sub_event_col] = df_local[sub_event_col].astype(str).str.strip()
    df_local[gov_col] = df_local[gov_col].astype(str).str.strip()

    analysis_df = df_local[
        df_local[event_col].isin(["protests", "riots"])
        & df_local[sub_event_col].isin(TARGET_SUBTYPES)
        & df_local[gov_col].ne("")
        & df_local[gov_col].str.lower().ne("nan")
    ].copy()

    if analysis_df.empty:
        print(f"{dataset_label}: no valid protests/riots rows after filtering.")
        return None

    grouped = (
        analysis_df.groupby([gov_col, sub_event_col], dropna=False)
        .size()
        .rename("event_count")
        .reset_index()
    )

    wide = grouped.pivot_table(
        index=gov_col,
        columns=sub_event_col,
        values="event_count",
        fill_value=0,
    ).reset_index()

    for subtype in TARGET_SUBTYPES:
        if subtype not in wide.columns:
            wide[subtype] = 0

    wide["total_events"] = wide[TARGET_SUBTYPES].sum(axis=1)

    for subtype in TARGET_SUBTYPES:
        pct_col = f"{subtype} (%)"
        wide[pct_col] = ((wide[subtype] / wide["total_events"]) * 100).round(2)

    pct_cols = [f"{s} (%)" for s in TARGET_SUBTYPES]
    wide = wide.sort_values("total_events", ascending=False).reset_index(drop=True)

    long_df = wide[[gov_col] + pct_cols].melt(
        id_vars=gov_col,
        value_vars=pct_cols,
        var_name="Subtype",
        value_name="Percent",
    )
    long_df["Subtype"] = long_df["Subtype"].str.replace(" (%)", "", regex=False)

    return {
        "label": dataset_label,
        "gov_col": gov_col,
        "table": wide,
        "long_df": long_df,
        "pct_cols": pct_cols,
    }


def plot_governorate_subtype_percentages(result):
    if result is None:
        return

    label = result["label"]
    gov_col = result["gov_col"]
    table = result["table"]
    long_df = result["long_df"]
    pct_cols = result["pct_cols"]

    if label == "ACLED_egypt2011":
        period_label = "Arab Spring"
    elif label == "ACLED_alsisi":
        period_label = "2019-2020"
    else:
        period_label = label

    fig = px.bar(
        long_df,
        x=gov_col,
        y="Percent",
        color="Subtype",
        category_orders={"Subtype": TARGET_SUBTYPES},
        color_discrete_map=SUBTYPE_COLORS,
        barmode="stack",
        title=f"Demonstration Events by Governorate in % ({period_label})",
    )

    fig.update_layout(
        xaxis_title="Governorate",
        yaxis_title="Percent of events",
        yaxis_range=[0, 100],
        legend_title_text="Subtype",
        xaxis_tickangle=-40,
        height=680,
        margin=dict(l=0, r=0, t=70, b=0),
    )
    fig.update_traces(hovertemplate="Governorate: %{x}<br>Subtype: %{fullData.name}<br>Percent: %{y:.2f}%<extra></extra>")
    fig.show()

    summary_cols = [gov_col, "total_events"] + pct_cols
    print(f"\n{label} governorate subtype percentages:")
    display(table[summary_cols])


gov_pct_2011 = build_governorate_subtype_percentages(df_2011_clean, "ACLED_egypt2011")
gov_pct_alsisi = build_governorate_subtype_percentages(df_alsisi_clean, "ACLED_alsisi")

plot_governorate_subtype_percentages(gov_pct_2011)
plot_governorate_subtype_percentages(gov_pct_alsisi)


ACLED_egypt2011 governorate subtype percentages:


sub_event_type,admin1,total_events,Peaceful protest (%),Excessive force against protesters (%),Violent demonstration (%),Protest with intervention (%)
0,Cairo,1158.0,71.68,2.68,20.38,5.27
1,Giza,204.0,74.51,2.94,17.16,5.39
2,Alexandria,171.0,71.93,2.34,23.39,2.34
3,Gharbia,86.0,54.65,1.16,40.70,3.49
4,Suez,86.0,62.79,3.49,27.91,5.81
5,North Sinai,74.0,72.97,1.35,24.32,1.35
6,Port Said,58.0,58.62,5.17,32.76,3.45
7,Assiut,45.0,57.78,2.22,31.11,8.89
8,Dakahlia,40.0,55.00,0.00,42.50,2.50
9,Sharqia,39.0,69.23,0.00,25.64,5.13



ACLED_alsisi governorate subtype percentages:


sub_event_type,admin1,total_events,Peaceful protest (%),Excessive force against protesters (%),Violent demonstration (%),Protest with intervention (%)
0,Giza,46.0,39.13,6.52,10.87,43.48
1,Cairo,20.0,50.00,0.00,0.00,50.00
2,Beheira,8.0,62.50,0.00,12.50,25.00
3,Alexandria,8.0,62.50,12.50,12.50,12.50
4,Gharbia,8.0,100.00,0.00,0.00,0.00
5,Sharqia,6.0,83.33,0.00,0.00,16.67
6,Menia,5.0,20.00,0.00,0.00,80.00
7,Kafr el-Sheikh,5.0,100.00,0.00,0.00,0.00
8,Aswan,5.0,60.00,0.00,0.00,40.00
9,Luxor,5.0,20.00,0.00,20.00,60.00


In [34]:
import pandas as pd
import plotly.express as px

TARGET_SUBTYPES = [
    "Peaceful protest",
    "Excessive force against protesters",
    "Violent demonstration",
    "Protest with intervention",
]

SUBTYPE_COLORS = {
    "Peaceful protest": "#ff7f0e",
    "Excessive force against protesters": "#808080",
    "Violent demonstration": "#d62728",
    "Protest with intervention": "#000000",
}


def build_governorate_subtype_counts(df_in, dataset_label):
    if df_in is None or df_in.empty:
        print(f"{dataset_label}: no data available.")
        return None

    df_local = df_in.copy()
    norm_cols = {c.strip().lower(): c for c in df_local.columns}
    pick = lambda *names: next((norm_cols[n] for n in names if n in norm_cols), None)

    event_col = pick("event_type")
    sub_event_col = pick("sub_event_type")
    gov_col = pick("admin1", "governorate", "admin_1", "location")

    if not event_col or not sub_event_col or not gov_col:
        print(f"{dataset_label}: required columns missing for governorate analysis.")
        return None

    df_local[event_col] = df_local[event_col].astype(str).str.strip().str.lower()
    df_local[sub_event_col] = df_local[sub_event_col].astype(str).str.strip()
    df_local[gov_col] = df_local[gov_col].astype(str).str.strip()

    analysis_df = df_local[
        df_local[event_col].isin(["protests", "riots"])
        & df_local[sub_event_col].isin(TARGET_SUBTYPES)
        & df_local[gov_col].ne("")
        & df_local[gov_col].str.lower().ne("nan")
    ].copy()

    if analysis_df.empty:
        print(f"{dataset_label}: no valid protests/riots rows after filtering.")
        return None

    grouped = (
        analysis_df.groupby([gov_col, sub_event_col], dropna=False)
        .size()
        .rename("event_count")
        .reset_index()
    )

    wide = grouped.pivot_table(
        index=gov_col,
        columns=sub_event_col,
        values="event_count",
        fill_value=0,
    ).reset_index()

    for subtype in TARGET_SUBTYPES:
        if subtype not in wide.columns:
            wide[subtype] = 0

    wide["total_events"] = wide[TARGET_SUBTYPES].sum(axis=1)

    # Sort by total events
    wide = wide.sort_values("total_events", ascending=False).reset_index(drop=True)

    long_df = wide.melt(
        id_vars=gov_col,
        value_vars=TARGET_SUBTYPES,
        var_name="Subtype",
        value_name="Count",
    )

    return {
        "label": dataset_label,
        "gov_col": gov_col,
        "table": wide,
        "long_df": long_df,
        "value_cols": TARGET_SUBTYPES,
    }


def plot_governorate_subtype_counts(result):
    if result is None:
        return

    label = result["label"]
    gov_col = result["gov_col"]
    table = result["table"]
    long_df = result["long_df"]
    value_cols = result["value_cols"]

    if label == "ACLED_egypt2011":
        period_label = "Arab Spring"
    elif label == "ACLED_alsisi":
        period_label = "2019-2020"
    else:
        period_label = label

    fig = px.bar(
        long_df,
        x=gov_col,
        y="Count",
        color="Subtype",
        category_orders={"Subtype": TARGET_SUBTYPES},
        color_discrete_map=SUBTYPE_COLORS,
        barmode="stack",
        title=f"Demonstration Events by Governorate ({period_label})",
    )

    fig.update_layout(
        xaxis_title="Governorate",
        yaxis_title="Number of events",
        legend_title_text="Subtype",
        xaxis_tickangle=-40,
        height=680,
        margin=dict(l=0, r=0, t=70, b=0),
    )
    fig.update_traces(hovertemplate="Governorate: %{x}<br>Subtype: %{fullData.name}<br>Count: %{y}<extra></extra>")
    fig.show()

    summary_cols = [gov_col, "total_events"] + value_cols
    print(f"\n{label} governorate subtype counts:")
    display(table[summary_cols])


gov_counts_2011 = build_governorate_subtype_counts(df_2011_clean, "ACLED_egypt2011")
gov_counts_alsisi = build_governorate_subtype_counts(df_alsisi_clean, "ACLED_alsisi")

plot_governorate_subtype_counts(gov_counts_2011)
plot_governorate_subtype_counts(gov_counts_alsisi)


ACLED_egypt2011 governorate subtype counts:


sub_event_type,admin1,total_events,Peaceful protest,Excessive force against protesters,Violent demonstration,Protest with intervention
0,Cairo,1158.0,830.0,31.0,236.0,61.0
1,Giza,204.0,152.0,6.0,35.0,11.0
2,Alexandria,171.0,123.0,4.0,40.0,4.0
3,Gharbia,86.0,47.0,1.0,35.0,3.0
4,Suez,86.0,54.0,3.0,24.0,5.0
5,North Sinai,74.0,54.0,1.0,18.0,1.0
6,Port Said,58.0,34.0,3.0,19.0,2.0
7,Assiut,45.0,26.0,1.0,14.0,4.0
8,Dakahlia,40.0,22.0,0.0,17.0,1.0
9,Sharqia,39.0,27.0,0.0,10.0,2.0



ACLED_alsisi governorate subtype counts:


sub_event_type,admin1,total_events,Peaceful protest,Excessive force against protesters,Violent demonstration,Protest with intervention
0,Giza,46.0,18.0,3.0,5.0,20.0
1,Cairo,20.0,10.0,0.0,0.0,10.0
2,Beheira,8.0,5.0,0.0,1.0,2.0
3,Alexandria,8.0,5.0,1.0,1.0,1.0
4,Gharbia,8.0,8.0,0.0,0.0,0.0
5,Sharqia,6.0,5.0,0.0,0.0,1.0
6,Menia,5.0,1.0,0.0,0.0,4.0
7,Kafr el-Sheikh,5.0,5.0,0.0,0.0,0.0
8,Aswan,5.0,3.0,0.0,0.0,2.0
9,Luxor,5.0,1.0,0.0,1.0,3.0


## Comparison Across Same Governorates (2011 vs al sisi)
This chart compares subtype percentages for governorates that exist in both datasets.

In [35]:
import pandas as pd
import plotly.express as px

if "gov_pct_2011" not in globals() or "gov_pct_alsisi" not in globals() or gov_pct_2011 is None or gov_pct_alsisi is None:
    print("Run Cell 43 first to create governorate percentage tables.")
else:
    t2011 = gov_pct_2011["table"].copy()
    talsisi = gov_pct_alsisi["table"].copy()
    gov_col_2011 = gov_pct_2011["gov_col"]
    gov_col_alsisi = gov_pct_alsisi["gov_col"]

    pct_cols = [f"{s} (%)" for s in TARGET_SUBTYPES]

    # Keep only governorates present in BOTH datasets for direct comparison.
    common_governorates = sorted(set(t2011[gov_col_2011]).intersection(set(talsisi[gov_col_alsisi])))
    if not common_governorates:
        print("No common governorates found between datasets.")
    else:
        d2011 = t2011[t2011[gov_col_2011].isin(common_governorates)][[gov_col_2011] + pct_cols].copy()
        d2011 = d2011.rename(columns={gov_col_2011: "Governorate"})
        d2011["Dataset"] = "Arab Spring"

        dalsisi = talsisi[talsisi[gov_col_alsisi].isin(common_governorates)][[gov_col_alsisi] + pct_cols].copy()
        dalsisi = dalsisi.rename(columns={gov_col_alsisi: "Governorate"})
        dalsisi["Dataset"] = "2019-2020"

        combined = pd.concat([d2011, dalsisi], ignore_index=True)

        long_comp = combined.melt(
            id_vars=["Governorate", "Dataset"],
            value_vars=pct_cols,
            var_name="Subtype",
            value_name="Percent",
        )
        long_comp["Subtype"] = long_comp["Subtype"].str.replace(" (%)", "", regex=False)

        # One single stacked bar per panel (dataset x governorate).
        long_comp["Bar"] = "Composition"

        fig = px.bar(
            long_comp,
            x="Bar",
            y="Percent",
            color="Subtype",
            barmode="stack",
            facet_row="Governorate",
            facet_col="Dataset",
            facet_col_spacing=0.04,
            facet_row_spacing=0.02,
            category_orders={
                "Subtype": TARGET_SUBTYPES,
                "Dataset": ["Arab Spring", "2019-2020"],
                "Governorate": common_governorates,
            },
            color_discrete_map=SUBTYPE_COLORS,
            title="Temporal Composition of Demonstrations by Governorate (%)",
            height=max(900, 220 * len(common_governorates)),
        )

        # Remove default facet labels and collect positions.
        kept_annotations = []
        governorate_annotations = []
        dataset_x = {}
        for ann in fig.layout.annotations:
            if ann.text.startswith("Dataset="):
                ds = ann.text.replace("Dataset=", "")
                dataset_x[ds] = ann.x
            elif ann.text.startswith("Governorate="):
                governorate_annotations.append(ann)
            else:
                kept_annotations.append(ann)
        fig.layout.annotations = tuple(kept_annotations)

        # Explicit top headers so they are always visible above the two columns.
        x_arab = dataset_x.get("Arab Spring", 0.24)
        x_alsisi = dataset_x.get("2019-2020", 0.76)
        fig.add_annotation(
            x=x_arab,
            y=1.003,
            xref="paper",
            yref="paper",
            yanchor="bottom",
            text="Arab Spring",
            showarrow=False,
            font=dict(size=12),
        )
        fig.add_annotation(
            x=x_alsisi,
            y=1.003,
            xref="paper",
            yref="paper",
            yanchor="bottom",
            text="2019-2020",
            showarrow=False,
            font=dict(size=12),
        )

        # Add each governorate as a small centered title above its two panels.
        yaxis_domains = []
        for key in fig.layout:
            if key.startswith("yaxis") and getattr(fig.layout[key], "domain", None):
                dom = tuple(fig.layout[key].domain)
                if dom not in yaxis_domains:
                    yaxis_domains.append(dom)

        for ann in governorate_annotations:
            gov_name = ann.text.replace("Governorate=", "")
            y_center = ann.y
            row_domain = next((d for d in yaxis_domains if d[0] <= y_center <= d[1]), None)
            y_top = row_domain[1] + 0.003 if row_domain else y_center + 0.005
            fig.add_annotation(
                x=0.5,
                y=y_top,
                xref="paper",
                yref="paper",
                text=gov_name,
                showarrow=False,
                font=dict(size=11),
            )

        # Show fixed percentage ticks and avoid repeating the y-axis title on right panels.
        fig.update_yaxes(range=[0, 100], tickmode="array", tickvals=[0, 50, 100], title_text="Percent of events")

        for key in fig.layout:
            if key.startswith("yaxis"):
                axis_number = int(key[5:]) if key != "yaxis" else 1
                if axis_number % 2 == 0:
                    fig.layout[key].title.text = ""

        fig.update_xaxes(showticklabels=False, title_text="")
        fig.update_layout(
            legend_title_text="Subtype",
            margin=dict(l=0, r=0, t=150, b=20),
        )
        fig.update_traces(
            hovertemplate="Governorate: %{customdata[0]}<br>Dataset: %{customdata[1]}<br>Subtype: %{fullData.name}<br>Percent: %{y:.2f}%<extra></extra>",
            customdata=long_comp[["Governorate", "Dataset"]].values,
        )
        fig.show()

        print(f"Compared common governorates: {len(common_governorates)}")
        display(pd.DataFrame({"Governorate": common_governorates}))

Compared common governorates: 22


,Governorate
0,Alexandria
1,Assiut
2,Aswan
3,Beheira
4,Cairo
5,Dakahlia
6,Damietta
7,Fayoum
8,Gharbia
9,Giza


In [49]:
# Comparison across selected governorates only: Giza, Cairo, Beheira
import pandas as pd
import plotly.express as px

selected_governorates = ["Giza", "Cairo", "Beheira"]

if "gov_pct_2011" not in globals() or "gov_pct_alsisi" not in globals() or gov_pct_2011 is None or gov_pct_alsisi is None:
    print("Run the governorate percentage cells first to create gov_pct_2011 and gov_pct_alsisi.")
else:
    t2011 = gov_pct_2011["table"].copy()
    talsisi = gov_pct_alsisi["table"].copy()
    gov_col_2011 = gov_pct_2011["gov_col"]
    gov_col_alsisi = gov_pct_alsisi["gov_col"]

    pct_cols = [f"{s} (%)" for s in TARGET_SUBTYPES]

    common_governorates = sorted(
        set(t2011[gov_col_2011]).intersection(set(talsisi[gov_col_alsisi]))
    )
    isolated_governorates = [
        gov for gov in selected_governorates if gov in common_governorates
    ]

    if not isolated_governorates:
        print("None of Giza, Cairo, or Beheira were found in both datasets.")
    else:
        d2011 = t2011[t2011[gov_col_2011].isin(isolated_governorates)][[gov_col_2011] + pct_cols].copy()
        d2011 = d2011.rename(columns={gov_col_2011: "Governorate"})
        d2011["Dataset"] = "Arab Spring"

        dalsisi = talsisi[talsisi[gov_col_alsisi].isin(isolated_governorates)][[gov_col_alsisi] + pct_cols].copy()
        dalsisi = dalsisi.rename(columns={gov_col_alsisi: "Governorate"})
        dalsisi["Dataset"] = "2019-2020"

        combined = pd.concat([d2011, dalsisi], ignore_index=True)

        long_comp = combined.melt(
            id_vars=["Governorate", "Dataset"],
            value_vars=pct_cols,
            var_name="Subtype",
            value_name="Percent",
        )
        long_comp["Subtype"] = long_comp["Subtype"].str.replace(" (%)", "", regex=False)

        long_comp["Bar"] = "Composition"

        fig = px.bar(
            long_comp,
            x="Bar",
            y="Percent",
            color="Subtype",
            barmode="stack",
            facet_row="Governorate",
            facet_col="Dataset",
            facet_col_spacing=0.04,
            facet_row_spacing=0.02,
            category_orders={
                "Subtype": TARGET_SUBTYPES,
                "Dataset": ["Arab Spring", "2019-2020"],
                "Governorate": isolated_governorates,
            },
            color_discrete_map=SUBTYPE_COLORS,
            title="Temporal Composition of Demonstrations by Governorate (%) - Giza, Cairo, Beheira",
            height=max(700, 220 * len(isolated_governorates)),
        )

        kept_annotations = []
        governorate_annotations = []
        dataset_x = {}
        for ann in fig.layout.annotations:
            if ann.text.startswith("Dataset="):
                ds = ann.text.replace("Dataset=", "")
                dataset_x[ds] = ann.x
            elif ann.text.startswith("Governorate="):
                governorate_annotations.append(ann)
            else:
                kept_annotations.append(ann)
        fig.layout.annotations = tuple(kept_annotations)

        x_arab = dataset_x.get("Arab Spring", 0.24)
        x_alsisi = dataset_x.get("2019-2020", 0.76)
        fig.add_annotation(
            x=x_arab,
            y=1.003,
            xref="paper",
            yref="paper",
            yanchor="bottom",
            text="Arab Spring",
            showarrow=False,
            font=dict(size=12),
        )
        fig.add_annotation(
            x=x_alsisi,
            y=1.003,
            xref="paper",
            yref="paper",
            yanchor="bottom",
            text="2019-2020",
            showarrow=False,
            font=dict(size=12),
        )

        yaxis_domains = []
        for key in fig.layout:
            if key.startswith("yaxis") and getattr(fig.layout[key], "domain", None):
                dom = tuple(fig.layout[key].domain)
                if dom not in yaxis_domains:
                    yaxis_domains.append(dom)

        for ann in governorate_annotations:
            gov_name = ann.text.replace("Governorate=", "")
            y_center = ann.y
            row_domain = next((d for d in yaxis_domains if d[0] <= y_center <= d[1]), None)
            y_top = row_domain[1] + 0.003 if row_domain else y_center + 0.005
            fig.add_annotation(
                x=0.5,
                y=y_top,
                xref="paper",
                yref="paper",
                text=gov_name,
                showarrow=False,
                font=dict(size=11),
            )

        fig.update_yaxes(range=[0, 100], tickmode="array", tickvals=[0, 50, 100], title_text="Percent of events")

        for key in fig.layout:
            if key.startswith("yaxis"):
                axis_number = int(key[5:]) if key != "yaxis" else 1
                if axis_number % 2 == 0:
                    fig.layout[key].title.text = ""

        fig.update_xaxes(showticklabels=False, title_text="")
        fig.update_layout(
            legend_title_text="Subtype",
            margin=dict(l=0, r=0, t=150, b=20),
        )
        fig.update_traces(
            hovertemplate="Governorate: %{customdata[0]}<br>Dataset: %{customdata[1]}<br>Subtype: %{fullData.name}<br>Percent: %{y:.2f}%<extra></extra>",
            customdata=long_comp[["Governorate", "Dataset"]].values,
        )
        fig.show()

        print(f"Compared isolated governorates: {', '.join(isolated_governorates)}")
        display(pd.DataFrame({"Governorate": isolated_governorates}))

Compared isolated governorates: Giza, Cairo, Beheira


,Governorate
0,Giza
1,Cairo
2,Beheira


## Governorate Protest Quantity Comparison (2011 vs Al Sisi)
This chart displays the total number of protest and riot events per governorate across both datasets, allowing direct comparison of protest activity intensity.

In [36]:
import pandas as pd
import plotly.express as px

TARGET_SUBTYPES = [
    "Peaceful protest",
    "Excessive force against protesters",
    "Violent demonstration",
    "Protest with intervention",
]


def build_governorate_quantity_comparison(df_2011, df_alsisi):
    """
    Build a comparison of protest quantities (total event counts) by governorate across both datasets.
    """
    results = {}
    
    for df_in, dataset_label in [(df_2011, "ACLED_egypt2011"), (df_alsisi, "ACLED_alsisi")]:
        if df_in is None or df_in.empty:
            print(f"{dataset_label}: no data available.")
            continue
        
        df_local = df_in.copy()
        norm_cols = {c.strip().lower(): c for c in df_local.columns}
        pick = lambda *names: next((norm_cols[n] for n in names if n in norm_cols), None)
        
        event_col = pick("event_type")
        sub_event_col = pick("sub_event_type")
        gov_col = pick("admin1", "governorate", "admin_1", "location")
        
        if not event_col or not sub_event_col or not gov_col:
            print(f"{dataset_label}: required columns missing.")
            continue
        
        df_local[event_col] = df_local[event_col].astype(str).str.strip().str.lower()
        df_local[sub_event_col] = df_local[sub_event_col].astype(str).str.strip()
        df_local[gov_col] = df_local[gov_col].astype(str).str.strip()
        
        analysis_df = df_local[
            df_local[event_col].isin(["protests", "riots"])
            & df_local[sub_event_col].isin(TARGET_SUBTYPES)
            & df_local[gov_col].ne("")
            & df_local[gov_col].str.lower().ne("nan")
        ].copy()
        
        if analysis_df.empty:
            print(f"{dataset_label}: no valid protests/riots rows after filtering.")
            continue
        
        # Count total events per governorate
        gov_counts = (
            analysis_df.groupby(gov_col, dropna=False)
            .size()
            .rename("count")
            .reset_index()
        )
        
        results[dataset_label] = {
            "gov_col": gov_col,
            "data": gov_counts,
        }
    
    return results


# Build quantities for both datasets
qty_results = build_governorate_quantity_comparison(df_2011_clean, df_alsisi_clean)

if not qty_results or len(qty_results) < 2:
    print("Not enough data to build comparison.")
else:
    gov_col_2011 = qty_results["ACLED_egypt2011"]["gov_col"]
    gov_col_alsisi = qty_results["ACLED_alsisi"]["gov_col"]
    
    d_2011 = qty_results["ACLED_egypt2011"]["data"].copy()
    d_2011 = d_2011.rename(columns={gov_col_2011: "Governorate", "count": "Arab Spring"})
    
    d_alsisi = qty_results["ACLED_alsisi"]["data"].copy()
    d_alsisi = d_alsisi.rename(columns={gov_col_alsisi: "Governorate", "count": "2019-2020"})
    
    # Merge on common governorates
    merged = pd.merge(d_2011, d_alsisi, on="Governorate", how="outer").fillna(0)
    merged["Arab Spring"] = merged["Arab Spring"].astype(int)
    merged["2019-2020"] = merged["2019-2020"].astype(int)
    merged["Total"] = merged["Arab Spring"] + merged["2019-2020"]
    merged["Difference"] = merged["Arab Spring"] - merged["2019-2020"]
    
    # Sort by total quantity descending
    merged = merged.sort_values("Total", ascending=False).reset_index(drop=True)
    
    # Reshape for plotly: convert wide format to long for grouped bars
    long_qty = merged[["Governorate", "Arab Spring", "2019-2020"]].melt(
        id_vars="Governorate",
        value_vars=["Arab Spring", "2019-2020"],
        var_name="Dataset",
        value_name="Quantity",
    )
    
    # Create grouped bar chart
    fig = px.bar(
        long_qty,
        x="Governorate",
        y="Quantity",
        color="Dataset",
        barmode="group",
        title="Demonstration Events by Governorate: Arab Spring vs 2019-2020",
        color_discrete_map={
            "Arab Spring": "#d62728",
            "2019-2020": "#1f77b4",
        },
        labels={"Quantity": "Number of Events"},
    )
    
    fig.update_layout(
        xaxis_title="Governorate",
        yaxis_title="Number of Events",
        xaxis_tickangle=-45,
        height=600,
        margin=dict(l=0, r=0, t=70, b=100),
        legend_title_text="Dataset",
        hovermode="x unified",
    )
    fig.update_traces(
        hovertemplate="Governorate: %{x}<br>Dataset: %{fullData.name}<br>Events: %{y}<extra></extra>"
    )
    fig.show()
    
    # Display summary table sorted by total quantity
    print("\nProtest Quantity by Governorate (sorted by total):")
    display(merged[["Governorate", "Arab Spring", "2019-2020", "Total", "Difference"]])
    print(f"\nTotal governorates with data: {len(merged)}")
    print(f"Total events (Arab Spring): {merged['Arab Spring'].sum()}")
    print(f"Total events (2019-2020): {merged['2019-2020'].sum()}")



Protest Quantity by Governorate (sorted by total):


,Governorate,Arab Spring,2019-2020,Total,Difference
0,Cairo,1158,20,1178,1138
1,Giza,204,46,250,158
2,Alexandria,171,8,179,163
3,Gharbia,86,8,94,78
4,Suez,86,3,89,83
5,North Sinai,74,1,75,73
6,Port Said,58,1,59,57
7,Assiut,45,1,46,44
8,Sharqia,39,6,45,33
9,Dakahlia,40,4,44,36



Total governorates with data: 27
Total events (Arab Spring): 2323
Total events (2019-2020): 147


In [38]:
import pandas as pd
import plotly.express as px

# Composition of protests vs riots by dataset, shown in percent only.
def build_event_type_share(df_2011, df_alsisi):
    frames = []

    for df_in, dataset_label in [(df_2011, "Arab Spring"), (df_alsisi, "2019-2020")]:
        if df_in is None or df_in.empty:
            continue

        df_local = df_in.copy()
        norm_cols = {c.strip().lower(): c for c in df_local.columns}
        pick = lambda *names: next((norm_cols[n] for n in names if n in norm_cols), None)

        event_col = pick("event_type")
        date_col = pick("event_date")

        if not event_col or not date_col:
            print(f"{dataset_label}: required columns missing.")
            continue

        df_local[event_col] = df_local[event_col].astype(str).str.strip().str.lower()
        df_local[date_col] = pd.to_datetime(df_local[date_col], errors="coerce")
        df_local = df_local.dropna(subset=[date_col]).copy()
        df_local = df_local[df_local[event_col].isin(["protests", "riots"])].copy()

        counts = df_local[event_col].value_counts()
        total = counts.sum()

        frames.append(
            pd.DataFrame(
                {
                    "Dataset": dataset_label,
                    "Event type": ["protests", "riots"],
                    "Count": [int(counts.get("protests", 0)), int(counts.get("riots", 0))],
                    "Percent": [
                        round((counts.get("protests", 0) / total) * 100, 1) if total else 0,
                        round((counts.get("riots", 0) / total) * 100, 1) if total else 0,
                    ],
                }
            )
        )

    if not frames:
        return pd.DataFrame(columns=["Dataset", "Event type", "Count", "Percent"])

    return pd.concat(frames, ignore_index=True)

composition = build_event_type_share(df_egypt2011_clean, df_alsisi_clean)

if composition.empty:
    print("No composition data available.")
else:
    share_table = composition.pivot(index="Dataset", columns="Event type", values="Percent").fillna(0)
    share_table = share_table[["protests", "riots"]].rename(columns={"protests": "Protests (%)", "riots": "Riots (%)"})
    display(share_table)

    fig = px.bar(
        composition,
        x="Dataset",
        y="Percent",
        color="Event type",
        barmode="group",
        text="Percent",
        color_discrete_map={"protests": "#1f77b4", "riots": "#d62728"},
        title="Protests vs Riots in Egypt by Dataset (%)",
        labels={"Percent": "Percentage of events", "Dataset": "Dataset"},
    )
    fig.update_traces(textposition="outside", hovertemplate="Dataset: %{x}<br>Event type: %{fullData.name}<br>Percent: %{y:.1f}%<extra></extra>")
    fig.update_layout(yaxis_range=[0, 100], legend_title_text="Event type", height=500, margin=dict(l=40, r=40, t=70, b=40))
    fig.show()


Event type,Protests (%),Riots (%)
Dataset,,
2019-2020,93.2,6.8
Arab Spring,69.4,30.6


In [39]:
# Peaceful protests vs Violent demonstrations (% by dataset)
# Computes counts and % of total events per dataset, displays a table and renders an inline bar chart + saves an HTML chart.
import pandas as pd
import plotly.express as px

def peaceful_vs_violent_percent(df, name):
    total = len(df)
    sub_col = 'sub_event_type' if 'sub_event_type' in df.columns else ('SUB_EVENT_TYPE' if 'SUB_EVENT_TYPE' in df.columns else None)
    if sub_col is None:
        peaceful = 0
        violent = 0
    else:
        peaceful = int(df[sub_col].str.contains('Peaceful protest', case=False, na=False).sum())
        violent = int(df[sub_col].str.contains('Violent demonstration', case=False, na=False).sum())
    pct_peaceful = round((peaceful / total * 100),1) if total>0 else None
    pct_violent = round((violent / total * 100),1) if total>0 else None
    return dict(dataset=name, total_events=total, peaceful=peaceful, violent=violent, peaceful_pct=pct_peaceful, violent_pct=pct_violent)

r1 = peaceful_vs_violent_percent(df_egypt2011_clean, 'Arab Spring (2011-2013)')
r2 = peaceful_vs_violent_percent(df_alsisi_clean, '2019-2020')
out = pd.DataFrame([r1, r2])

print('\nPeaceful protests vs Violent demonstrations (% of total events)')
print(out[['dataset','total_events','peaceful','violent','peaceful_pct','violent_pct']].to_string(index=False))

# Plot inline and save
plot_df = out.melt(id_vars=['dataset'], value_vars=['peaceful_pct','violent_pct'], var_name='type', value_name='pct')
plot_df['type'] = plot_df['type'].map({'peaceful_pct':'Peaceful protests (% of events)','violent_pct':'Violent demonstrations (% of events)'})
fig = px.bar(plot_df, x='dataset', y='pct', color='type', barmode='group', text='pct', labels={'pct':'% of events','dataset':'Dataset','type':''})
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(yaxis=dict(range=[0,100]), legend_title_text=None, title='Peaceful vs Violent (% of total events)')
# Show inline
fig.show()
# Save static HTML copy
output_html = 'peaceful_vs_violent_by_dataset.html'
fig.write_html(output_html, include_plotlyjs='cdn')
print(f"\nChart saved to: {output_html}")



Peaceful protests vs Violent demonstrations (% of total events)
                dataset  total_events  peaceful  violent  peaceful_pct  violent_pct
Arab Spring (2011-2013)          2564      1611      543          62.8         21.2
              2019-2020           147        77       10          52.4          6.8



Chart saved to: peaceful_vs_violent_by_dataset.html
